In [ ]:
%load_ext autoreload
%autoreload 2

import functools
import itertools
from math import pi
from multiprocessing.pool import Pool
import os
from typing import Any, NamedTuple


os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from numpy.typing import ArrayLike, NDArray
import scipy
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.graph import (
    create_rotated_surface_code,
    create_hexagon_code,
    create_heavy_hexagon_code,
    plot_graph,
)
from fluxoniumcr.qubits.fluxonium import Fluxonium

# Functions

## Fluxonium eigenvalue lookup

In [ ]:
bare_fx_dataset = xr.load_dataset(DATA_DIR/"fluxonium_quantization"/"dim=8_cutoff=128"/"dataset.hdf5")
driven_fx_dataset = xr.load_dataset(DATA_DIR/"fluxonium_quasienergy_precomp"/"dataset.hdf5")

driven_fx_dataset_intp = scipy.interpolate.RegularGridInterpolator(
    (
        driven_fx_dataset.EJ.data,
        driven_fx_dataset.EL.data,
        driven_fx_dataset.drive_frequency.data,
    ),
    driven_fx_dataset.quasienergy.data.squeeze(),
    bounds_error=False,
)
bare_fx_dataset_intp = scipy.interpolate.RegularGridInterpolator(
    (
        bare_fx_dataset.ELdivEJ.data,
        bare_fx_dataset.EBdivEC.data,
    ),
    bare_fx_dataset.eigenenergy.data,
    bounds_error=False,
)


def get_driven_fluxonium_eigenvalues(
        EJ: ArrayLike,
        EC: ArrayLike,
        EL: ArrayLike,
        drive_frequency: ArrayLike,
        deltap: float,
) -> tuple[NDArray[np.floating], NDArray[np.floating]]:
    assert np.isclose(EC, driven_fx_dataset.EC.item(), atol=1e-9)
    
    points = np.array(np.broadcast_arrays(EJ, EL, drive_frequency)).T
    values = driven_fx_dataset_intp(points)
    
    evals_bare = values[..., 0, :]
    evals_driven = values[..., abs(driven_fx_dataset.deltap.data - deltap).argmin(), :]

    return evals_bare, evals_driven


def get_bare_fluxonium_eigenvalues(
        EJ: ArrayLike,
        EC: ArrayLike,
        EL: ArrayLike,
) -> NDArray[np.floating]:
    EJ, EC, EL = np.broadcast_arrays(EJ, EC, EL)
    out_shape = EJ.shape
    
    ELdivEJ = EL/EJ
    EJdivEB = calculate_EJdivEB(ELdivEJ)
    EBdivEC = EJ/EJdivEB/EC
    
    EC = np.broadcast_to(EC, out_shape)
    points = np.array(
        np.broadcast_arrays(
            np.ravel(ELdivEJ),
            np.ravel(EBdivEC),
        )
    ).T
    
    evals = bare_fx_dataset_intp(points)
    evals *= np.ravel(EC)[..., None]
    evals = evals.reshape(*out_shape, -1)
    evals = evals[..., :4]  # Keep only first 4 levels.
    
    # Manually calculate if not within precomputed domain.
    mask = np.isnan(evals[..., 0])
    if np.any(mask):
        EJ_masked = EJ[mask]
        EC_masked = EC[mask]
        EL_masked = EL[mask]
        fx = Fluxonium(
            EJ=EJ_masked,
            EC=EC_masked,
            EL=EL_masked,
            dim=4,
            cutoff=32,
        )
        evals[mask] = fx.eigenvalues
    
    return evals


def calculate_EJdivEB(ratio):
    a = sinc_inv(ratio)
    return 1/(1 - a * np.sin(a)/2 - np.cos(a))


def sinc_inv(y):
    x = np.pi*(1 - y)
    # 10 newton iterations should be enough.
    for _ in range(10):
        x = x - (sinc(x) - y)/(dsinc(x) + 1e-6)
    return x


def sinc(x):
    return np.sinc(x/np.pi)


def dsinc(x):
    return np.piecewise(
        x,
        [abs(x) > 0],
        [
            lambda x: (np.cos(x) - sinc(x))/x,
            0.0,
        ]
    )

## Random fluxonium sampler

In [ ]:
class DrivenFluxoniumSpectrum(NamedTuple):
    fd: Any
    f10_bare: Any
    f10: Any
    f21: Any
    f20: Any
    f30: Any

        
class BareFluxoniumSpectrum(NamedTuple):
    f10: Any
    f21: Any
        

class ControlFluxoniumEnsemble:
    def __init__(
            self,
            EJ: float,
            EC: float,
            EL: float,
            EJ_rel_sdev: float,
            EL_rel_sdev: float,
            size: int = 2**10,
            deltap: float = 0.8,
    ) -> None:
        self.EJ = EJ
        self.EC = EC
        self.EL = EL
        self.EJ_rel_sdev = EJ_rel_sdev
        self.EL_rel_sdev = EL_rel_sdev
        self.deltap = deltap
        self.size = size
        
        self.EJ_values, self.EL_values = generate_random_EJ_and_EL(
            EJ,
            EL,
            EJ_rel_sdev,
            EL_rel_sdev,
            size,
        )
        
    def __call__(
            self,
            drive_frequency: float,
    ) -> DrivenFluxoniumSpectrum:
        spectrum = get_driven_fluxonium_spectrum(
            EJ=self.EJ_values,
            EC=self.EC,
            EL=self.EL_values,
            drive_frequency=drive_frequency,
            deltap=self.deltap,
        )
        
        return spectrum

    
def generate_target_ensemble(
        EJ: float,
        EC: float,
        EL: float,
        EJ_rel_sdev: float,
        EL_rel_sdev: float,
        size: int,
):
    EJ_values, EL_values = generate_random_EJ_and_EL(
        EJ,
        EL,
        EJ_rel_sdev,
        EL_rel_sdev,
        size,
    )

    spectrum = get_bare_fluxonium_spectrum(
        EJ=EJ_values,
        EC=EC,
        EL=EL_values,
    )

    return spectrum
    

def generate_random_EJ_and_EL(
        EJ: float,
        EL: float,
        EJ_rel_sdev: float,
        EL_rel_sdev: float,
        size: int,
) -> tuple[NDArray[np.floating], NDArray[np.floating]]:
    EJ_std = EJ * EJ_rel_sdev
    EJ_values = np.random.normal(
        loc=EJ,
        scale=EJ_std,
        size=size,
    )
    EL_std = EL * EL_rel_sdev
    EL_values = np.random.normal(
        loc=EL,
        scale=EL_std,
        size=size,
    )
    return EJ_values, EL_values


def get_driven_fluxonium_spectrum(
        EJ: ArrayLike,
        EC: ArrayLike,
        EL: ArrayLike,
        drive_frequency: ArrayLike,
        deltap: float,
) -> DrivenFluxoniumSpectrum:
    evals_bare, evals_driven = get_driven_fluxonium_eigenvalues(
        EJ,
        EC,
        EL,
        drive_frequency,
        deltap,
    )
    
    f10_bare = evals_bare[..., 1] - evals_bare[..., 0]
    f10 = evals_driven[..., 1] - evals_driven[..., 0]
    f21 = evals_driven[..., 2] - evals_driven[..., 1]
    f20 = evals_driven[..., 2] - evals_driven[..., 0]
    f30 = evals_driven[..., 3] - evals_driven[..., 0]
    
    return DrivenFluxoniumSpectrum(
        fd=drive_frequency,
        f10_bare=f10_bare,
        f10=f10,
        f21=f21,
        f20=f20,
        f30=f30,
    )


def get_bare_fluxonium_spectrum(
        EJ: ArrayLike,
        EC: ArrayLike,
        EL: ArrayLike,
) -> BareFluxoniumSpectrum:
    evals = get_bare_fluxonium_eigenvalues(EJ, EC, EL)
    
    f10 = evals[..., 1] - evals[..., 0]
    f21 = evals[..., 2] - evals[..., 1]
    
    return BareFluxoniumSpectrum(
        f10=f10,
        f21=f21,
    )

## Frequency collision checker

In [ ]:
def calculate_zero_collision_probability(
        G: nx.Graph,
        EJ_nominal: ArrayLike,
        EC_nominal: ArrayLike,
        EL_nominal: ArrayLike,
        EJ_rel_sdev: float,
        EL_rel_sdev: float,
        ensemble_size: int,
) -> float:
    EJ_nominal, EC_nominal, EL_nominal = np.broadcast_arrays(
        EJ_nominal,
        EC_nominal,
        EL_nominal,
    )
    
    node_colors = nx.get_node_attributes(G, 'color')

    target_spectra = {}
    for i, j in G.nodes:
        color = node_colors[(i, j)]
        if color == 0:
            # Skip control qubits
            continue
        target_spectra[(i, j)] = generate_target_ensemble(
            EJ_nominal[color],
            EC_nominal[color],
            EL_nominal[color],
            EJ_rel_sdev,
            EL_rel_sdev,
            ensemble_size,
        )

    collisions = np.zeros(ensemble_size)
    for i, j in G.nodes:
        color = node_colors[(i, j)]
        if color != 0:
            # Skip target qubits
            continue

        control_ensemble = ControlFluxoniumEnsemble(
            EJ_nominal[0],
            EC_nominal[0],
            EL_nominal[0],
            EJ_rel_sdev,
            EL_rel_sdev,
            ensemble_size,
        )
        control_spectra = {
            n: control_ensemble(target_spectra[n].f10)
            for n in G.neighbors((i, j))
        }

        for spec in control_spectra.values():
            collisions += check_control_collisions(spec).sum(axis=0)

        for n1, n2 in itertools.product(
                control_spectra.keys(),
                repeat=2,
        ):
            collisions += check_control_and_target_collisions(
                control_spectra[n1],
                target_spectra[n2],
            ).sum(axis=0)

        for n1, n2 in itertools.combinations(
                control_spectra.keys(),
                2,
        ):
            collisions += check_control_and_spectator_collisions(
                control_spectra[n1],
                target_spectra[n1],
                control_spectra[n2],
                target_spectra[n2],
            ).sum(axis=0)
    
    return (collisions == 0).mean()


def check_control_collisions(control_spec: DrivenFluxoniumSpectrum) -> NDArray[np.bool]:
    collisions = np.array([
        (
            ((control_spec.f10_bare - control_spec.fd) > -80e-3 * 2*pi)
            | np.isnan(control_spec.f10_bare)
        ),
        (
            abs(control_spec.f21 - 3*control_spec.fd) < 60e-3 * 2*pi
        ),
        (
            abs(control_spec.f20 - 4*control_spec.fd) < 80e-3 * 2*pi
        ),
        (
            abs(control_spec.f30 - 5*control_spec.fd) < 25e-3 * 2*pi
        ),
    ])
    
    return collisions


def check_control_and_target_collisions(
        control_spec: DrivenFluxoniumSpectrum,
        target_spec: BareFluxoniumSpectrum,
) -> NDArray[np.bool]:
    collisions = np.array([
        (
            abs(target_spec.f21 - 3*control_spec.fd) < 15e-3 * 2*pi
        ),
        (
            ((delta := target_spec.f21 - control_spec.f10 - 2*control_spec.fd)
                 > -20e-3 * 2*pi)
            & (delta < +40e-3 * 2*pi)
        ),
        (
            ((delta := control_spec.f21 - target_spec.f10 - 2*control_spec.fd)
                 > -20e-3 * 2*pi)
            & (delta < +10e-3 * 2*pi)
        ),
    ])
    
    return collisions


def check_control_and_spectator_collisions(
        control1_spec: DrivenFluxoniumSpectrum,
        target1_spec: BareFluxoniumSpectrum,
        control2_spec: DrivenFluxoniumSpectrum,
        target2_spec: BareFluxoniumSpectrum,
) -> NDArray[np.bool]:
    collisions = np.array([
        (
            abs(target1_spec.f10 - target2_spec.f10) < 35e-3 * 2*pi
        ),
        (
            ((delta := target2_spec.f10 + control1_spec.f10 - 2*control1_spec.fd)
                 > -40e-3 * 2*pi)
            & (delta < +20e-3 * 2*pi)
        ),
        (
            ((delta := target1_spec.f10 + control2_spec.f10 - 2*control2_spec.fd)
                 > -40e-3 * 2*pi)
            & (delta < +20e-3 * 2*pi)
        ),
    ])
    
    return collisions

# Create dataset

In [ ]:
ensemble_size = 2**12
repetition_data = np.arange(2**8)
EJ_rel_sdev_data = np.geomspace(0.005, 0.1, 100)
distance_data = np.array([3, 5, 7, 21], dtype=np.int8)

dataset = xr.Dataset(
    attrs=dict(
        ensemble_size=ensemble_size
    )
)

repetition_coord = xr.DataArray(
    repetition_data,
    dims=['repetition']
)
EJ_rel_sdev_coord = xr.DataArray(
    EJ_rel_sdev_data,
    dims=['EJ_rel_sdev']
)
qubit_coord = xr.DataArray(
    np.arange(5, dtype=np.int8),
    dims=['qubit'],
)
lattice_coord = xr.DataArray(
    ["square", "hexagon", "heavyhex"],
    dims=['lattice'],
)
distance_coord = xr.DataArray(
    distance_data,
    dims=['distance'],
)

dataset['EL_rel_sdev'] = xr.DataArray(
    # Assume EL varies 10x less than EJ.
    EJ_rel_sdev_data/10,
    coords=[EJ_rel_sdev_coord],
)
dataset['num_qubits'] = xr.DataArray(
    np.int32(0),
    coords=[lattice_coord, distance_coord],
)
dataset['EJ'] = xr.DataArray(
    np.nan,
    coords=[lattice_coord, qubit_coord],
    attrs=dict(
        unit="Grad/s"
    )   
)
dataset['EC'] = xr.DataArray(
    np.nan,
    coords=[lattice_coord, qubit_coord],
    attrs=dict(
        unit="Grad/s"
    )   
)
dataset['EL'] = xr.DataArray(
    np.nan,
    coords=[lattice_coord, qubit_coord],
    attrs=dict(
        unit="Grad/s"
    )   
)
dataset['zero_collision_prob'] = xr.DataArray(
    np.nan,
    coords=[lattice_coord, distance_coord, repetition_coord, EJ_rel_sdev_coord],
)

## Define nominal EJ, EC, EL parameters

In [ ]:
dataset.EC[:] = 2*pi * np.array([1.2] + [1.0] * 4)
dataset.EL[:] = 2*pi * np.array([0.4] + [1.0] * 4)

ds = dataset.loc[dict(lattice='square')]
ds.EJ[:] = 2*pi * np.array([4.0, 4.6, 3.7, 3.4, 3.1])

ds = dataset.loc[dict(lattice='hexagon')]
ds.EJ[:4] = 2*pi * np.array([4.0, 4.5, 3.5, 3.1])
ds.EC[4:] = np.nan
ds.EL[4:] = np.nan

ds = dataset.loc[dict(lattice='heavyhex')]
ds.EJ[:3] = 2*pi * np.array([4.0, 4.4, 3.2])
ds.EC[3:] = np.nan
ds.EL[3:] = np.nan

# Do calculations

In [ ]:
graph_factories = {
    'square': create_rotated_surface_code,
    'hexagon': create_hexagon_code,
    'heavyhex': create_heavy_hexagon_code,
}

with tqdm.auto.tqdm(
        total=(
            len(dataset.lattice.data)
            * len(dataset.distance.data)
            * len(dataset.repetition.data)
        ),
) as pbar, Pool(processes=8) as pool:
    for lattice, distance in itertools.product(
            dataset.lattice.data,
            dataset.distance.data,
    ):
        ds2 = dataset.loc[dict(lattice=lattice, distance=distance)]
        G = graph_factories[lattice](distance)
        ds2['num_qubits'][()] = len(G.nodes)
        for repetition in dataset.repetition.data:
            ds1 = ds2.loc[dict(repetition=repetition)]
            mask = np.isnan(ds1['zero_collision_prob'])
            ds1_masked = ds1.sel(EJ_rel_sdev=mask)
            for EJ_rel_sdev, prob in zip(
                    ds1_masked.EJ_rel_sdev.data,
                    pool.starmap(
                        functools.partial(
                            calculate_zero_collision_probability,
                            G,
                            ds1.EJ.data,
                            ds1.EC.data,
                            ds1.EL.data,
                            ensemble_size=ds1.attrs['ensemble_size'],
                        ),
                        zip(
                            ds1_masked.EJ_rel_sdev.data,
                            ds1_masked.EL_rel_sdev.data,
                        )
                    )
            ):
                ds = ds1.loc[dict(EJ_rel_sdev=EJ_rel_sdev)]
                ds['zero_collision_prob'][()] = prob
            pbar.update(1)

In [ ]:
parent_path = DATA_DIR/"zero_collision_yield"
parent_path.mkdir(parents=True, exist_ok=True)
dataset.to_netcdf(parent_path/"dataset.hdf5")